# FYM on the DiTFlow DAVIS50 benchmark (Colab)

Protocol: `steps.md`. Rows per clip: `backbone` (clean Wan2.1-T2V-1.3B) and `fym` (per-clip spatial + temporal LoRA tuning).
The claim is the paired per-clip `fym - backbone` with a bootstrap CI.

Every session: cells 1-5. Then the pilot (6-8) or the full run (9-11).
Results live on Drive, so a recycled runtime loses nothing: re-run cells 1-5 and re-issue the run; finished cells are skipped.

Runtime: **A100 40GB, High-RAM**.

In [3]:
# 1. Paths and pins
DRIVE    = "/content/drive/MyDrive/ditflow"
RUNS     = f"{DRIVE}/runs"                    # run tree; collect.py reads it
BENCH    = "/content/bench"                   # tars unpacked here each session
MODELS   = "/content/models/Wan2.1-T2V-1.3B"  # local: tuning reads the weights several times per clip
REPO     = "/content/FollowYourMotion"
GIT_URL  = "https://github.com/jnheinrich451-eng/FollowYourMotion.git"
BRANCH   = "davis-bench"
WAN_REPO = "Wan-AI/Wan2.1-T2V-1.3B"
# DiTFlow scored with CoTracker unpinned (hub 'main'). main has not moved since this commit
# (2025-01-21) and the cotracker3 weights not since 2024-10, so this is the tracker it used.
COTRACKER_REF = "82e02e8029753ad4ef13cf06be7f4fc5facdda4d"

In [4]:
# 2. Drive, GPU, RAM -- stops here on the wrong runtime rather than partway through a run
from google.colab import drive
drive.mount("/content/drive")
import subprocess, os
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip()
ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
print(f"GPU: {gpu} | RAM: {ram_gb:.0f} GB")
assert "A100" in gpu, "not an A100: create a new Colab server with the A100 accelerator and reconnect"
assert ram_gb > 40, "standard RAM: pick High-RAM (the text encoder alone is ~11 GB in CPU RAM)"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-40GB, 40960 MiB | RAM: 83 GB


In [5]:
# 3. Code and dependencies (Colab's own torch is used; transformers pinned to the repo's
#    requirements.txt, since the 5.x Colab ships breaks diffsynth's imports)
import os
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} {GIT_URL} {REPO}
!git -C {REPO} pull -q; git -C {REPO} log --oneline -1
!pip install -q transformers==4.46.2 lightning==2.6.1 peft==0.18.1 modelscope ftfy sentencepiece einops imageio imageio-ffmpeg lpips git+https://github.com/openai/CLIP.git
!python -c "import transformers; print('transformers', transformers.__version__)"

ee2529f (HEAD -> davis-bench, origin/davis-bench) Stop the notebook early on a non-A100 or standard-RAM runtime
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 155.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.4/236.4 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [6]:
# 4. Benchmark data: unpack every tar on Drive, find the packed root and the masks, remap the manifest
import glob, pathlib, subprocess
os.makedirs(BENCH, exist_ok=True)
for t in sorted(glob.glob(f"{DRIVE}/*.tar")):
    marker = pathlib.Path(BENCH) / (".unpacked_" + pathlib.Path(t).name)
    if not marker.exists():
        print("unpacking", t)
        subprocess.run(["tar", "-xf", t, "-C", BENCH], check=True)
        marker.touch()

metas = [p for p in pathlib.Path(BENCH).rglob("camel/meta.json") if p.parents[1].name == "davis"]
masks = list(pathlib.Path(BENCH).rglob("camel/00000.png"))
assert metas, "no davis/camel/meta.json under BENCH: is the packed tar on Drive?"
assert masks, "no camel/00000.png under BENCH: is the annotations tar on Drive?"
PACKED, ANNOT = str(metas[0].parents[2]), str(masks[0].parents[1])
MANIFEST = "/content/davis50_local.csv"
print("packed:", PACKED, "| annotations:", ANNOT)
subprocess.run(["python", "benchmark/remap.py", "--manifest", "benchmark/davis50.csv", "--out", MANIFEST,
                "--from", "E:\\bench\\packed", "--to", PACKED, "--verify"], cwd=REPO, check=True)

unpacking /content/drive/MyDrive/ditflow/canon.tar
unpacking /content/drive/MyDrive/ditflow/davis_annotations.tar
unpacking /content/drive/MyDrive/ditflow/packed.tar
packed: /content/bench/packed | annotations: /content/bench/davis_annotations


CompletedProcess(args=['python', 'benchmark/remap.py', '--manifest', 'benchmark/davis50.csv', '--out', '/content/davis50_local.csv', '--from', 'E:\\bench\\packed', '--to', '/content/bench/packed', '--verify'], returncode=0)

In [7]:
# 5. Wan2.1-T2V-1.3B, pinned: the first session records the revision on Drive, later sessions reuse it
from huggingface_hub import HfApi, snapshot_download
pin = pathlib.Path(DRIVE) / "fym_wan_REVISION"
rev = pin.read_text().strip() if pin.exists() else HfApi().model_info(WAN_REPO).sha
pin.write_text(rev)
if not os.path.exists(f"{MODELS}/diffusion_pytorch_model.safetensors"):
    snapshot_download(WAN_REPO, revision=rev, local_dir=MODELS)
pathlib.Path(MODELS, "REVISION").write_text(f"{WAN_REPO}@{rev}")
print(f"{WAN_REPO}@{rev}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

Wan-AI/Wan2.1-T2V-1.3B@37ec512624d61f7aa208f7ea8140a131f93afc9a


## Pilot: `camel`, both rows

Also runs `fym_biasfix` (the pretrained q/k/v biases kept through the head split). It checks whether the release's random biases matter; it is not a headline row. Leave it out of the full run unless the pilot says it matters.

Check before the full run:
- `temporal heads per layer` varies across layers (not all 0 or all 12);
- the contact sheet: backbone and fym show the new subject, and fym moves like the reference;
- the scores table has a value in every column.

In [8]:
# 6. Pilot run
!cd {REPO} && python fym_bench/run_davis.py --manifest {MANIFEST} --runs {RUNS} --models {MODELS} --clips camel --configs backbone,fym,fym_biasfix

1 clips x ['backbone', 'fym', 'fym_biasfix']; checkpoint Wan-AI/Wan2.1-T2V-1.3B@37ec512624d61f7aa208f7ea8140a131f93afc9a; repo ee2529f334dd19c7e926a89b4d5163e98c417f92; gpu NVIDIA A100-SXM4-40GB
[1/1] camel
    backbone: FAILED at generate (missing_dep: exit code 1)
    fym: tuning
    fym: FAILED at data_process (missing_dep: exit code 1)
    fym_biasfix: tuning
    fym_biasfix: FAILED at data_process (missing_dep: exit code 1)
  elapsed so far 1 min

accounting: {
  "backbone": {
    "attempted": 1,
    "completed": 0,
    "failed": {
      "missing_dep": 1
    }
  },
  "fym": {
    "attempted": 1,
    "completed": 0,
    "failed": {
      "missing_dep": 1
    }
  },
  "fym_biasfix": {
    "attempted": 1,
    "completed": 0,
    "failed": {
      "missing_dep": 1
    }
  }
}


In [11]:
# 7. Pilot: what happened (a failed cell prints its stage, reason and log tail)
import json
from IPython.display import Image as show
for cfg in ["backbone", "fym", "fym_biasfix"]:
    d = pathlib.Path(RUNS, "fym_wan/davis/camel/subject", cfg, "seed1")
    if (d / "done.json").exists():
        j = json.loads((d / "done.json").read_text())
        print(f"{cfg}: done", {k: j.get(k) for k in ("tune_s", "elapsed_s", "peak_gpu_mb")})
    elif (d / "failed.json").exists():
        j = json.loads((d / "failed.json").read_text())
        print(f"{cfg}: FAILED at {j['stage']} ({j['reason_code']}: {j['reason']})")
        for line in j["log_tail"].strip().splitlines()[-15:]:
            print("   ", line)
    else:
        print(f"{cfg}: not run")
    if (d / "tune/tune.json").exists():
        t = json.loads((d / "tune/tune.json").read_text())
        print("   temporal heads per layer:", t["temporal_heads_per_layer"])
        print("   stage seconds:", t["stage_s"])
!cd {REPO} && python fym_bench/contact_sheet.py --runs {RUNS} --clips camel --configs backbone,fym,fym_biasfix --out {DRIVE}/sheets_pilot
sheet = f"{DRIVE}/sheets_pilot/sheet_01.png"
show(sheet) if os.path.exists(sheet) else print("no contact sheet: no finished cells yet")

backbone: FAILED at generate (missing_dep: exit code 1)
    Traceback (most recent call last):
      File "/content/FollowYourMotion/fym_bench/generate.py", line 142, in <module>
        main()
        ~~~~^^
      File "/content/FollowYourMotion/fym_bench/generate.py", line 103, in main
        from diffsynth import ModelManager, WanVideoPipeline_Override
      File "/content/FollowYourMotion/diffsynth/__init__.py", line 3, in <module>
        from .prompters import *
      File "/content/FollowYourMotion/diffsynth/prompters/__init__.py", line 11, in <module>
        from .stepvideo_prompter import StepVideoPrompter
      File "/content/FollowYourMotion/diffsynth/prompters/stepvideo_prompter.py", line 3, in <module>
        from ..models.stepvideo_text_encoder import STEP1TextEncoder
      File "/content/FollowYourMotion/diffsynth/models/stepvideo_text_encoder.py", line 21, in <module>
        from transformers.modeling_utils import PretrainedConfig, PreTrainedModel
    ImportError: c

In [10]:
# 8. Pilot: score it
import pandas as pd
!cd {REPO} && python benchmark/collect.py --runs {RUNS} --manifest {MANIFEST} --annotations {ANNOT} --out {DRIVE}/scores_fym_pilot.csv --cotracker-ref {COTRACKER_REF} --scratch ""
pd.read_csv(f"{DRIVE}/scores_fym_pilot.csv")

Traceback (most recent call last):
  File "/content/FollowYourMotion/benchmark/collect.py", line 510, in <module>
    main()
    ~~~~^^
  File "/content/FollowYourMotion/benchmark/collect.py", line 347, in main
    cells = sorted(p.parent for p in Path(args.runs).rglob("done.json"))
  File "/content/FollowYourMotion/benchmark/collect.py", line 347, in <genexpr>
    cells = sorted(p.parent for p in Path(args.runs).rglob("done.json"))
                                     ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "/usr/lib/python3.13/glob.py", line 505, in select_recursive
    yield from select_recursive_step(stack, match_pos)
  File "/usr/lib/python3.13/glob.py", line 535, in select_recursive_step
    stack.append(entry_path)
    ~~~~~~~~~~~~^^^^^^^^^^^^
KeyboardInterrupt
^C


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ditflow/scores_fym_pilot.csv'

## Full run: all 50 clips

Runs in the background with its log on Drive, so a dropped VS Code connection does not stop it. Re-issue cell 9 after a recycled runtime and it resumes where it stopped.

In [ ]:
# 9. Full run, in the background
LOG = f"{DRIVE}/fym_run.log"
!cd {REPO} && nohup python fym_bench/run_davis.py --manifest {MANIFEST} --runs {RUNS} --models {MODELS} --configs backbone,fym >> {LOG} 2>&1 &
print("log:", LOG)

In [ ]:
# 10. Progress (re-run any time)
!tail -n 25 {LOG}
!cat {RUNS}/fym_wan/accounting.json 2>/dev/null

In [ ]:
# 11. Score, C1 floor, paired contrasts (all clips, then single / multi subject), contact sheets
import pandas as pd
SCORES = f"{DRIVE}/scores_fym.csv"
!cd {REPO} && python benchmark/collect.py --runs {RUNS} --manifest {MANIFEST} --annotations {ANNOT} --out {SCORES} --cotracker-ref {COTRACKER_REF} --scratch ""
!cd {REPO} && python benchmark/collect.py --runs {RUNS} --manifest {MANIFEST} --annotations {ANNOT} --out {SCORES} --cotracker-ref {COTRACKER_REF} --scratch "" --control c1
!cd {REPO} && python benchmark/analyse.py --scores {SCORES} --reference backbone --out {DRIVE}/tables_fym.md
s = pd.read_csv(SCORES)
for group in ["single", "multi"]:
    s[s["subjects"] == group].to_csv(f"/content/scores_fym_{group}.csv", index=False)
    !cd {REPO} && python benchmark/analyse.py --scores /content/scores_fym_{group}.csv --reference backbone --out {DRIVE}/tables_fym_{group}.md
!cd {REPO} && python fym_bench/contact_sheet.py --runs {RUNS} --configs backbone,fym --out {DRIVE}/sheets